# Feature Engineering

Extraire des features à partir des champs bruts : fréquence de connexion par IP, nombre de ports distincts, ratio requêtes/erreurs, entropie des URLs. Documenter chaque feature.

## Importation des librairies

In [12]:
import pandas as pd
import numpy as np
import math
import warnings
warnings.filterwarnings('ignore')

In [13]:
chemin_CICIDS = '../Data/cicids_clean.csv'
chemin_UNSW   = '../Data/unsw_clean.csv'
chemin_LOGS   = '../Data/logs_clean.csv'

df_cicids = pd.read_csv(chemin_CICIDS, low_memory=False)
df_unsw   = pd.read_csv(chemin_UNSW,   low_memory=False)
df_logs   = pd.read_csv(chemin_LOGS,   low_memory=False)



## Dataset 1 : CIC-IDS-2017



### 1.1 Ratio paquets envoyés / reçus

On cherche à observer le apport entre le nombre de paquets envoyés (forward) et reçus (backward) car une attaque DDoS génère des paquets dans un seul sens ainsi un ratio très élevé (beaucoup plus envoyés que reçus) est suspect.

In [14]:
df_cicids['ratio_paquets'] = df_cicids['Total Fwd Packets'] / (df_cicids['Total Backward Packets'] + 1) #on met 1 pour éviter la division par zéro
print(df_cicids['ratio_paquets'].describe().round(2))
print(df_cicids.groupby('Label')['ratio_paquets'].mean().round(2))

count    223080.00
mean          1.62
std           2.69
min           0.12
25%           0.50
50%           0.67
75%           2.00
max         245.00
Name: ratio_paquets, dtype: float64
Label
BENIGN    0.96
DDoS      2.12
Name: ratio_paquets, dtype: float64


On remarque que le ratio moyen est de 0.96 pour le trafic BENIGN contre 2.12 pour le trafic DDoS. Cela confirme que les attaques DDoS envoient en moyenne 2 fois plus de paquets 
qu'elles n'en reçoivent alors que le trafic normal est plutôt équilibré avec presque un ratio de 1.
De plus le maximum atteint est de 245 ce qui représente des flux déséquilibrés soit 245 paquets envoyés pour seulement 1 reçu .

### 1.2 Ratio octets envoyés / reçus

Ici on cherche à observer le rapport entre le volume de données envoyées (forward) et reçues (backward) en octets. Une attaque peut envoyer beaucoup de petits paquets ou peu de gros paquets.

In [15]:
df_cicids['ratio_octets'] = df_cicids['Total Length of Fwd Packets'] / (df_cicids['Total Length of Bwd Packets'] + 1)
print(df_cicids['ratio_octets'].describe().round(2))
print(df_cicids.groupby('Label')['ratio_octets'].mean().round(2))

count    223080.00
mean         31.02
std         273.82
min           0.00
25%           0.00
50%           0.25
75%          24.00
max       19897.00
Name: ratio_octets, dtype: float64
Label
BENIGN    59.49
DDoS       9.87
Name: ratio_octets, dtype: float64


Contrairement aux résultats prédédent ici c'est l'inverse.Le trafic BENIGN a un ratio moyen de 59.49 alors qu'on a seulement 9.87 pour le trafic DDoS.
Cela peut s'expliquer par le comportement DDoS car les attaques envoient des petits paquets en masse mais ne reçoivent quasiment rien en retour.
Alorsn que le trafic normal lui envoie moins de paquets mais de plus grande taille (pages web, fichiers...)c'est ce qui augmente le ratio.
Le maximum à 19 897 représente des transferts de fichiers volumineux dans un seul sens. Le ratio_octets montrevqu'un ratio élevé correspond ici à du trafic normal et pas à une attaque.


### 1.3 Débit moyen par paquet

La quantité moyenne de données transmises par paquet est le rapport entre le total d'octets sur le total paquets. Comme on l'a vu au dessus on sait que les attaques DDoS utilisent souvent de très petits paquets pour maximiser le nombre de connexions ainsi un débit très faible est suspect.

In [16]:
df_cicids['debit_par_paquet'] = (
    (df_cicids['Total Length of Fwd Packets'] + df_cicids['Total Length of Bwd Packets']) /(df_cicids['Total Fwd Packets'] + df_cicids['Total Backward Packets'] + 1))
print(df_cicids['debit_par_paquet'].describe().round(2))
print(df_cicids.groupby('Label')['debit_par_paquet'].mean().round(2))

count    223080.00
mean        515.38
std         562.26
min           0.00
25%           5.00
50%         120.00
75%        1162.70
max        1936.83
Name: debit_par_paquet, dtype: float64
Label
BENIGN    217.66
DDoS      736.47
Name: debit_par_paquet, dtype: float64


On remarque que le trafic  DDoS a un débit moyen par paquet de 736 octets contre 218 octets pour le trafic BENIGN.
Cela peut s'expliquer par la nature de cette attaque DDoS spécifique car ce dataset contient des attaques DDoS HTTP qui  envoient de vraies requêtes HTTP complètes donc assez lourdes plutôt que de simples paquets vides.
La médiane globale à 120 octets montre que la majorité des flux sont légers mais les DDoS HTTP tirent la moyenne vers le haut avec leurs requêtes volumineuses.


### 1.4 Indicateur de connexion courte vers port 80

On définit un indicateur binaire (0/1) signalant si la connexion dure moins d'1 seconde et cible le port 80 (HTTP).
Dans le fichier EDA_part3 on a observé qu'il pouvait s'agit d'une fenêtre suspecte car 9 connexions sur 10 respectant ce critère sont des attaques DDoS.

In [17]:
df_cicids['connexion_suspecte'] = ((df_cicids['Flow Duration'] < 1_000_000) &(df_cicids['Destination Port'] == 80)).astype(int)
print(df_cicids['connexion_suspecte'].value_counts())
print(df_cicids[df_cicids['connexion_suspecte'] == 1]['Label'].value_counts())

connexion_suspecte
0    175342
1     47738
Name: count, dtype: int64
Label
DDoS      43340
BENIGN     4398
Name: count, dtype: int64


On remarque que sur les 223 080 connexions 47 738 sont identifiées comme suspectes (connexion inférieur à 1s vers port 80) soit environ 21% du dataset.
Parmi ces connexions suspectes on compte 43 340 DDoS et 4 398 étant BENIGN.
Ainsi cette combinaison est bien un indicateur fiable.


## Dataset 2 : UNSW-NB15

### 2.1 Fréquence de connexion par IP source

On cherche à obtenir le nombre total de connexions initiées par chaque adresse IP source car une IP qui génère un très grand nombre de connexions est potentiellement en train de scanner le réseau ou de mener une attaque. C'est l'un des indicateurs les plus classiques de comportement malveillant.

In [18]:
freq_ip_src = df_unsw.groupby('srcip').size().reset_index(name='freq_connexion_src')
df_unsw = df_unsw.merge(freq_ip_src, on='srcip', how='left')
print(df_unsw['freq_connexion_src'].describe().round(0))
print(freq_ip_src.sort_values('freq_connexion_src', ascending=False).head())

count    700001.0
mean      62341.0
std       13977.0
min           1.0
25%       64640.0
50%       66145.0
75%       67091.0
max       67209.0
Name: freq_connexion_src, dtype: float64
         srcip  freq_connexion_src
32  59.166.0.2               67209
30  59.166.0.0               67128
35  59.166.0.5               67091
34  59.166.0.4               66722
31  59.166.0.1               66587


Ici la fréquence moyenne de connexion par IP est de 62 341 connexions ce qui est très élevé mais peut s'expliquer par le fait que le dataset UNSW-NB15 ne contient que 
quelques dizaines d'IP différentes donc chacun générant un très grand volume de trafic. Le Top 5 est dominé par les IP du sous-réseau 59.166.0. avec jusqu'à 67 209 connexions pour l'IP 59.166.0.2

### 2.2 Nombre de ports distincts contactés par IP source

De même ici on cherche le nombre de ports de destination uniques contactés par chaque IP source.


In [19]:
ports_distincts = df_unsw.groupby('srcip')['dsport'].nunique().reset_index(name='nb_ports_distincts')
df_unsw = df_unsw.merge(ports_distincts, on='srcip', how='left')
print(df_unsw['nb_ports_distincts'].describe().round(0))
print(df_unsw.groupby('Label')['nb_ports_distincts'].mean().round(1))

count    700001.0
mean      17338.0
std        4325.0
min           1.0
25%       18198.0
50%       18383.0
75%       18604.0
max       18986.0
Name: nb_ports_distincts, dtype: float64
Label
0    17895.6
1      324.2
Name: nb_ports_distincts, dtype: float64


On remarque que le trafic normal (Label 0) contacte en moyenne 17 896 ports distincts contre seulement 324 pour les attaques (Label 1).
Cela peut s'expliquer par la structure du dataset UNSW-NB15 en effet les IP normales génèrent un trafic très varié sur de longues périodes contactant 
naturellement beaucoup de ports différents tandis que les attaques sont ciblées sur des ports précis.
Le maximum à 18 986 ports distincts confirme que certaines IP légitimes couvrent presque l'ensemble des ports possibles (0-65535).
Dans ce dataset un faible nombre de ports distincts est donc un indicateur d'attaque. 

### 2.3 Nombre d'IP de destination distinctes par IP source

On cherche le nombre d'adresses IP de destination différentes contactées par chaque IP source.
Une IP qui contacte un très grand nombre de destinations différentes peut être en train de scanner tout un sous-réseau. Combinée au nombre de ports distincts cette feature permet de distinguer un scan de port ciblé d'un scan de réseau large.

In [20]:
nb_dst_distincts = df_unsw.groupby('srcip')['dstip'].nunique().reset_index(name='nb_dst_distincts')
df_unsw = df_unsw.merge(nb_dst_distincts, on='srcip', how='left')
print(df_unsw['nb_dst_distincts'].describe().round(0))
print(df_unsw.groupby('Label')['nb_dst_distincts'].mean().round(1))

count    700001.0
mean         10.0
std           1.0
min           1.0
25%          10.0
50%          10.0
75%          10.0
max          12.0
Name: nb_dst_distincts, dtype: float64
Label
0     9.9
1    11.4
Name: nb_dst_distincts, dtype: float64


La différence entre les classes est faible car le trafic normal contacte en moyenne 9.9 IP distinctes contre 11.4 pour les attaques.
La médiane étant à 10 avec un écart-type de 1 cela signifie que presque toutes les IP contactent le même nombre de destinations.

### 2.4 Ratio octets envoyés / reçus

On calcule le rapport entre les octets source (sbytes) et les octets destination (dbytes).


In [24]:
df_unsw['ratio_octets'] = df_unsw['sbytes'] / (df_unsw['dbytes'] + 1)

print(df_unsw['ratio_octets'].describe().round(2))
print(df_unsw.groupby('Label')['ratio_octets'].mean().round(2))

count    700001.00
mean         13.66
std         223.39
min           0.00
25%           0.11
50%           0.78
75%           1.05
max       33728.00
Name: ratio_octets, dtype: float64
Label
0      7.66
1    196.72
Name: ratio_octets, dtype: float64


Le ratio octets source/destination présente un écart très marqué entre les classes car on 7.66 pour le trafic normal contre 196.72 pour les attaques et donc confirme que les connexions malveillantes génèrent un trafic massivement unidirectionnel. 

### 2.5 Durée moyenne des connexions par IP source

On calcule la durée moyenne (en secondes) de toutes les connexions initiées par une IP source.
En effet les attaques génèrent souvent des connexions très courtes ou très longues ainsi la durée moyenne par IP permet de détecter ces comportements anormaux.

In [25]:
df_unsw['duree_moy_src'] = df_unsw.groupby('srcip')['dur'].transform('mean')
print(df_unsw['duree_moy_src'].describe().round(2))
print(df_unsw.groupby('Label')['duree_moy_src'].mean().round(2))

count    700001.00
mean          0.86
std           2.74
min           0.00
25%           0.61
50%           0.64
75%           0.66
max          45.85
Name: duree_moy_src, dtype: float64
Label
0    0.86
1    1.05
Name: duree_moy_src, dtype: float64


La durée moyenne des connexions est très concentrée entre 0.61s et 0.66s avec une médiane à 0.64s ainsi le comportement est assez homogène pour la majorité du trafic. L'écart entre les classes est faible avec 0.86s pour le trafic normal contre 1.05s pour les attaques mais le maximum à 45.85s et l'écart-type de 2.74 montrent l'existence de quelques connexions plutôt longues et donc potentiellement associées à des comportements de persistance. 

### 2.6 Ratio requêtes/erreurs

In [29]:
print(df_unsw['state'].value_counts())

state
FIN    487911
CON    187505
INT     21799
REQ      2429
CLO       111
URH       108
RST        74
ECO        26
ACC        22
PAR         4
TXD         2
URN         2
no          2
MAS         2
TST         2
ECR         2
Name: count, dtype: int64


INT sont les connexion jamais terminée proprement et RST les coupures brutales ainsi les deux sont typiques d'une attaque avortée.

In [30]:
df_unsw['ratio_echec'] = df_unsw.groupby('srcip')['state'].transform(lambda x: x.isin(['INT', 'RST']).sum() / len(x))
print(df_unsw['ratio_echec'].describe().round(2))
print(df_unsw.groupby('Label')['ratio_echec'].mean().round(2))

count    700001.00
mean          0.03
std           0.15
min           0.00
25%           0.00
50%           0.00
75%           0.00
max           1.00
Name: ratio_echec, dtype: float64
Label
0    0.01
1    0.54
Name: ratio_echec, dtype: float64


Ici l'écart entre classes est très important car on a  0.01 pour le trafic normal contre 0.54 pour les attaques soit plus d'une connexion sur deux en état INT ou RST côté malveillant. Cela confirme que les IP attaquantes génèrent énormément de connexions avortées ou alors brutalement interrompues.

## Dataset 3 : Cybersecurity Threat Detection Logs


### 3.1 Fréquence de connexion par IP source


In [31]:
df_logs['freq_connexion_src'] = df_logs.groupby('source_ip')['source_ip'].transform('count')
print(df_logs['freq_connexion_src'].describe().round(0))
print(df_logs.groupby('source_ip')['freq_connexion_src'].first().nlargest(5))

count    6000000.0
mean       16975.0
std          679.0
min        16142.0
25%        16495.0
50%        16611.0
75%        17867.0
max        18295.0
Name: freq_connexion_src, dtype: float64
source_ip
59.211.9.207       18295
109.106.120.222    18273
88.72.40.56        18252
185.225.185.68     18239
122.63.201.122     18229
Name: freq_connexion_src, dtype: int64


On remarque que la distribution est assez homogène on a toutes les IPs entre 16 142 et 18 295 connexions, avec une médiane à 16 611 et un écart-type de seulement 679.

### 3.2 Ratio requêtes bloquées sur le total par IP

Pour chaque IP source on cherche la proportion de ses requêtes qui ont été bloquées par rapport au total.
Une IP avec un taux de blocage élevé a déclenché de nombreuses règles de sécurité, c'est un fort indicateur de comportement malveillant.

In [33]:
df_logs['ratio_blocage'] = df_logs.groupby('source_ip')['action'].transform(lambda x: (x == 'blocked').sum() / len(x))
print(df_logs['ratio_blocage'].describe().round(2))
print(df_logs.groupby('threat_label')['ratio_blocage'].mean().round(2))

count    6000000.00
mean           0.50
std            0.00
min            0.49
25%            0.50
50%            0.50
75%            0.50
max            0.51
Name: ratio_blocage, dtype: float64
threat_label
benign        0.5
malicious     0.5
suspicious    0.5
Name: ratio_blocage, dtype: float64


Toutes les valeurs sont figées à 0.50 soient mean, médiane, et les trois classes (benign, malicious, suspicious) affichent exactement le même ratio

### 3.3 Entropie des URL par IP source

L'entropie mesure le niveau de désordre ou d'aléatoire dans un ensemble de données. Plus les valeurs sont variées et imprévisibles, plus l'entropie est élevée. Plus elles sont répétitives et prévisibles, plus elle est faible.
La formule de Shannon est : H = -Σ p(x) * log2(p(x)) où p(x) est la fréquence relative de chaque valeur. Un utilisateur normal visite :/home, /home, /home, /products, /home
Il y a eu peu de variété donc l'entropie sera faible alors qu'un attaquant génère :/home, /login?id=1, /search?q=UNION SELECT, /../../etc/passwd, /xyz123
Il y a beaucoup de variété donc l'entropie sera élevée.

In [36]:
def entropie_shannon(series):
    p = series.value_counts(normalize=True)  # fréquence relative de chaque URL
    return -(p * np.log2(p)).sum()           # formule de Shannon
df_logs['entropie_url'] = df_logs.groupby('source_ip')['request_path'].transform(entropie_shannon) # calcul et ajout direct au dataframe
print(df_logs['entropie_url'].describe().round(2))
print(df_logs.groupby('threat_label')['entropie_url'].mean().round(2))

count    6000000.00
mean           3.54
std            0.49
min            3.17
25%            3.22
50%            3.23
75%            4.26
max            4.35
Name: entropie_url, dtype: float64
threat_label
benign        3.50
malicious     4.01
suspicious    3.98
Name: entropie_url, dtype: float64


On observe un écart entre les classes on a 3.50 pour le trafic bénin contre 4.01 pour le trafic malveillant et 3.98 pour le suspicieux, ainsi cela confirme bien que les IP malveillantes visitent des URLs beaucoup plus diversifiées.

### 3.4 Entropie des caractères dans l'URL

Ici on fait de même mais avec l'entropie des caractères dans l'URL car une URL normale ressemble à ça :/home/products ainsi les caractères sont simples et répétitifs donc l' entropie est basse. Alors que dans une URL d'attaque on aura quelque chose du genre : /login?id=1' UNION SELECT * FROM users-- avec un mélange de lettres, chiffres et caractères dont l'entropie sera haute.

In [39]:
from collections import Counter  # Counter compte les occurrences de chaque élément

def entropie_caracteres(url):
    url = str(url)          # convertit en string 
    n = len(url)            # nombre total de caractères dans l'URL
    if n == 0:
        return 0            # pas de division par zéro si l'URL est vide
    counts = np.array(list(Counter(url).values()))  # compte chaque caractère et met sous forme de liste
    p = counts / n          # convertit en fréquences 
    return -(p * np.log2(p)).sum()  # applique la formule de Shannon 
df_logs['entropie_char_url'] = df_logs['request_path'].apply(entropie_caracteres) #on applique la fonction sur chaque URL du dataset
print(df_logs['entropie_char_url'].describe().round(2))                       
print(df_logs.groupby('threat_label')['entropie_char_url'].mean().round(2))    

count    6000000.00
mean           1.56
std            1.46
min           -0.00
25%           -0.00
50%            2.32
75%            2.81
max            4.39
Name: entropie_char_url, dtype: float64
threat_label
benign        1.40
malicious     3.48
suspicious    3.34
Name: entropie_char_url, dtype: float64


Ici on remarque un écart très marqué entre les classes avec 1.40 pour le trafic bénin contre 3.48 pour le malveillant et 3.34 pour le suspicieux. Cela confirme que les URLs d'attaque contiennent une grande variété de caractères spéciaux qui font monter l'entropie.

### 3.5 Longueur de l'URL

On cherche à observer le nombre de caractères dans l'URL de chaque requête car une URL anormalement longue est un signal d'alerte simple et efficace.

In [40]:
df_logs['longueur_url'] = df_logs['request_path'].str.len()
print(df_logs['longueur_url'].describe().round(0))
print(df_logs.groupby('threat_label')['longueur_url'].mean().round(1))

count    6000000.0
mean           6.0
std            5.0
min            1.0
25%            1.0
50%            5.0
75%           10.0
max           30.0
Name: longueur_url, dtype: float64
threat_label
benign         4.9
malicious     14.9
suspicious    14.3
Name: longueur_url, dtype: float64


La différence entre classes est très remarquable avec 4.9 caractères en moyenne pour le trafic bénin contre 14.9 pour le malveillant et 14.3 pour le suspicieux, soit environ 3 fois plus long. Ainsi les URLs normales sont courtes tandis que les URLs d'attaque sont plutôt longues.

## Sauvegarde

In [ ]:
df_cicids.to_csv('../Data/cicids_features.csv', index=False)
df_unsw.to_csv('../Data/unsw_features.csv', index=False)
df_logs.to_csv('../Data/logs_features.csv', index=False)


## Bilan des 3 datasets

Dataset 1 CICIDS2017 (4 features)
Ce dataset ne contient ni IP ni URL. Les 4 features créées ciblent spécifiquement les attaques DDoS HTTP où ratio_paquets et ratio_octets mesurent le déséquilibre directionnel du trafic, debit_par_paquet montre la nature volumineuse des requêtes HTTP d'attaque, ainsi que les connexion inférieure à 1s vers le port 80. 

Dataset 2 UNSW-NB15 (5 features)
freq_connexion_src, nb_ports_distincts et nb_dst_distincts permettent de détecter les comportements de scan réseau. ratio_echec (connexions INT et RST) s'est révélé assez discriminant avec 0.01 pour le trafic normal contre 0.54 pour les attaques.

Dataset 3 Cybersecurity Threat Detection Logs (5 features)
entropie_char_url est le feature le plus important avec 1.40 pour le bénin contre 3.48 pour le malveillant. longueur_url confirme ce signal avec des URL 3 fois plus longues pour les attaques. En revanche, freq_connexion_src et ratio_blocage se sont révélées peu importante à cause de la nature synthétique et équilibrée du dataset. 